# Predicting a pair with whole proteins held out


**Stage 5 of instruction 72.** Every other stage predicts a property of one gene; this predicts a property of two. The strategy ladder of instruction 73 says nothing about it, because all 69 of those questions are gene-level.

**Whole proteins are held out, not pairs.** Splitting pairs at random leaks: a protein in a hundred pairs can be memorised, and a held-out pair containing it then scores well without the model knowing anything about pairing. Genes are split by orthogroup and a pair is kept only when both endpoints land in the same partition, so a test pair has two unseen proteins.

**Popularity is the baseline to beat.** A well-studied, abundant protein appears in many recorded pairs, so the product of the two partner counts predicts well while knowing no biology. Matching the negatives on degree was tried and starves them, so the baseline is reported beside the model: not beating it is itself the answer.

Built 2026-11-09.

In [1]:
import json, os, numpy as np, pandas as pd
from parascape import datasets as ds, organisms as O, splits
pairs = pd.read_parquet(os.path.join(REPO, 'parascape/data/information_map/pairs.parquet'))
pairs = pairs[pairs.partner_side != 'host']
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
known = set(nodes.index)
pairs = pairs[pairs.gene_a.isin(known) & pairs.gene_b.isin(known)]
edge = pairs.apply(lambda r: tuple(sorted((r.gene_a, r.gene_b))), axis=1)
positives = pd.DataFrame(list(dict.fromkeys(edge)), columns=['a', 'b'])
print(len(pairs), 'recorded parasite-parasite pairs ->', len(positives), 'distinct pairs over',
      len(set(positives.a) | set(positives.b)), 'proteins')
degree = pd.concat([positives.a, positives.b]).value_counts()
degree.head(8).to_frame('pairs recorded')

36489 recorded parasite-parasite pairs -> 36433 distinct pairs over 7828 proteins


,pairs recorded
TGME49_263580,7095
TGME49_243440,7036
TGME49_218960,5983
TGME49_227290,4827
TGME49_304730,3406
TGME49_294860,2235
TGME49_233010,1168
TGME49_238895,937


## 1 · The split: whole proteins, by orthogroup

In [2]:
groups = nodes['orthogroup'].astype(object)
groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in nodes.index}))
entities = sorted(set(positives.a) | set(positives.b))
split = splits.make_split(entities, groups.reindex(entities).tolist(), organism=O.TOXOPLASMA,
                          benchmark_id='information_map_pairs_2026_11_09', seed=0)
where = {}
for part in ('train', 'tune', 'calibration', 'test'):
    for gene in split.entities(part):
        where[gene] = part
side = positives.assign(pa=positives.a.map(where), pb=positives.b.map(where))
kept = side[side.pa.notna() & (side.pa == side.pb)].copy()
kept['partition'] = kept.pa
print(len(kept), 'of', len(positives), 'pairs survive the cold-node rule (both ends same partition)')
kept.partition.value_counts().to_frame('pairs')

10367 of 36433 pairs survive the cold-node rule (both ends same partition)


,pairs
partition,
train,6561
calibration,2480
test,1117
tune,209


## 2 · Degree-matched negatives

For each positive pair, a negative pair is drawn inside the same partition whose two proteins have the same partition. Exact degree matching on both ends was tried and starves the negatives, so the popularity baseline of section 4 carries the weight instead: beating it is the only result that means anything, and not beating it is itself the answer.

In [3]:
rng = np.random.default_rng(0)
recorded = set(map(tuple, positives[['a', 'b']].to_numpy()))
# One negative per positive, drawn inside the same partition from the genes that appear in pairs
# at all. Exact degree matching on BOTH ends was tried first and starves: the high-degree bins hold
# too few genes, so only 9,021 of 13,540 pairs found a partner and the test set came out 94 %
# positive. Simple sampling keeps the balance, and the POPULARITY BASELINE in section 4 is then the
# number that matters -- a model that cannot beat the product of the two degrees has learnt
# popularity rather than pairing, and that is the finding either way.
rows = []
for part, block in kept.groupby('partition'):
    members = np.array(sorted(set(block.a) | set(block.b)))
    seen = set()
    for _ in range(len(block) * 30):
        if len(seen) >= len(block):
            break
        a, b = rng.choice(members, size=2, replace=False)
        key = tuple(sorted((str(a), str(b))))
        if key in recorded or key in seen:
            continue
        seen.add(key)
    for a, b in seen:
        rows.append(dict(a=a, b=b, partition=part, label=0))
negatives = pd.DataFrame(rows)
table = pd.concat([kept[['a', 'b', 'partition']].assign(label=1), negatives], ignore_index=True)
print(len(table), 'pairs:', int(table.label.sum()), 'recorded,', int((~table.label.astype(bool)).sum()), 'drawn')
pos_degree = pd.concat([kept.a, kept.b]).map(degree)
neg_degree = pd.concat([negatives.a, negatives.b]).map(degree)
pd.DataFrame({'recorded partners of a positive endpoint': pos_degree.describe(),
              'of a drawn endpoint': neg_degree.describe()}).round(2)

20734 pairs: 10367 recorded, 10367 drawn


,recorded partners of a positive endpoint,of a drawn endpoint
count,2.073e+04,2.073e+04
mean,2391,6.76
std,3040,63.58
min,1,1
25%,5,3
50%,19,4
75%,5983,6
max,7095,5983


## 3 · Symmetric pair features

A pair has no order, so each feature is a symmetric function of the two genes: the absolute difference, the smaller and the larger. The columns are the best-covered numeric measurements; `identity` columns are excluded, as everywhere.

In [4]:
from parascape import information_categories as IC
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c])
           and not pd.api.types.is_bool_dtype(nodes[c])]
def usable(c):
    cats = set(IC.categories_of(c, O.TOXOPLASMA))
    return bool(cats) and 'identity' not in cats
covered = [c for c in numeric if usable(c) and nodes[c].notna().mean() >= 0.5]
covered = sorted(covered, key=lambda c: -nodes[c].notna().mean())[:120]
A = nodes.loc[table.a, covered].to_numpy(dtype=float)
B = nodes.loc[table.b, covered].to_numpy(dtype=float)
X = pd.DataFrame(np.hstack([np.abs(A - B), np.minimum(A, B), np.maximum(A, B)]),
                 columns=[f'{k}:{c}' for k in ('diff', 'min', 'max') for c in covered])
y = table.label.to_numpy()
print(X.shape[1], 'pair features from', len(covered), 'gene columns')
X.iloc[:3, :4].round(3)

360 pair features from 120 gene columns


,diff:length,diff:n_tm,diff:has_signal_peptide,diff:is_tm
0,105,0,0,0
1,77,1,0,1
2,570,0,0,0


## 4 · Fit on train + tune, test on the cold pairs

The frozen test partition is touched once. The baselines are the ones that matter for pairs: the majority answer, and the product of the two proteins' recorded partner counts -- which is what a model would exploit if the negatives were not degree-matched.

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
fit = table.partition.isin(['train', 'tune']).to_numpy()
test = (table.partition == 'test').to_numpy()
model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                       min_samples_leaf=10, l2_regularization=1.0,
                                       early_stopping=False, random_state=0).fit(X[fit], y[fit])
score = model.predict_proba(X[test])[:, 1]
fame = (table.a.map(degree).fillna(1) * table.b.map(degree).fillna(1)).to_numpy()[test]
result = dict(train_pairs=int(fit.sum()), test_pairs=int(test.sum()),
              test_positive_share=round(float(y[test].mean()), 3),
              auroc=round(float(roc_auc_score(y[test], score)), 3),
              average_precision=round(float(average_precision_score(y[test], score)), 3),
              auroc_fame_baseline=round(float(roc_auc_score(y[test], fame)), 3),
              average_precision_fame=round(float(average_precision_score(y[test], fame)), 3))
json.dump(result, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print(result)
pd.DataFrame([result]).T.rename(columns={0: 'value'})

{'train_pairs': 13540, 'test_pairs': 2234, 'test_positive_share': 0.5, 'auroc': 0.913, 'average_precision': 0.833, 'auroc_fame_baseline': 0.998, 'average_precision_fame': 0.998}


,value
train_pairs,1.354e+04
test_pairs,2234
test_positive_share,0.5
auroc,0.913
average_precision,0.833
auroc_fame_baseline,0.998
average_precision_fame,0.998


## 5 · Did the fitness difference carry its weight?

The StarPath finding is the reason the absolute difference is in the features. Permutation importance over the test pairs, grouped by the kind of feature, says whether it did.

In [6]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(model, X[test], y[test], n_repeats=3, random_state=0,
                             scoring='average_precision')
weights = pd.Series(imp.importances_mean, index=X.columns)
by_kind = weights.groupby(weights.index.str.split(':').str[0]).sum().round(4)
top = weights.sort_values(ascending=False).head(12).round(4)
fitness = weights[[c for c in weights.index if c.split(':', 1)[1].startswith('fit_')]]
summary = dict(by_kind=by_kind.to_dict(),
               fitness_share=round(float(fitness.sum() / max(weights.clip(lower=0).sum(), 1e-9)), 3),
               fitness_difference_share=round(float(
                   fitness[[c for c in fitness.index if c.startswith('diff:')]].sum()
                   / max(weights.clip(lower=0).sum(), 1e-9)), 3))
json.dump(summary, open(os.path.join(OUT, 'importance.json'), 'w'), indent=1)
weights.sort_values(ascending=False).head(30).to_frame('importance').to_csv(
    os.path.join(OUT, 'importance.tsv'), sep='\t')
print(summary)
top.to_frame('importance')

{'by_kind': {'diff': 0.0061, 'max': 0.2667, 'min': 0.0426}, 'fitness_share': 0.0, 'fitness_difference_share': 0.0}


,importance
max:n_papers_focal,0.2576
min:n_xlink_partners,0.0453
min:restriction200962_kk18_p2wo_ph8_2_s1,0.0074
max:n_ipms_studies,0.0039
max:restriction200962_kk20_p2w_ph8_2_s16,0.003
diff:n_ipms_studies,0.0025
min:restriction200962_kk22_p5wo_ph8_2_s15,0.002
diff:te129869_confluent_r3,0.0019
diff:n_fulltext,0.0018
diff:n_bioid_studies,0.0017
